# Facet model: generating I-RAVEN answers on a per-puzzle object graph

The model **generates** the missing panel (per component: type, size and colour histograms, object count,
occupied positions) and then picks the candidate closest to what it generated; candidates are never seen
in training. Input: the ground-truth XML attributes of the 8 context panels. Every puzzle gets its own
graph, built by PyNeuraLogic's grounding from the objects that are actually there: no position nodes,
no padding.

Weights: `experiments/facet_all_w32-64-128_hidden` (all 7 constellations, 6000 training puzzles each, widths
32 → 64 → 128, lr 3e-4), the epoch with the best accuracy on val 0–299. Template, facts and evaluation
are imported from `pynl/bench_hub_grounding.py`, so this notebook runs exactly the trained code.

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")   # paths below are relative to the repo root
sys.path.insert(0, os.path.abspath("pynl"))
import neuralogic
neuralogic.set_max_memory_size(16)
from neuralogic.logging import clear_handlers; clear_handlers()
from neuralogic.core import R, Settings
from neuralogic.dataset import Dataset, Sample
from neuralogic.nn import Adam, MSE
import bench_hub_grounding as B          # facet template, facts, decoding and evaluation

DATA, CKPT = "data/I-RAVEN-10000", "experiments/facet_all_w32-64-128_hidden"
best = json.load(open(f"{CKPT}/best.json")); args = best["args"]
WIDTHS = [int(w) for w in args["widths"].split(",")]
print(f"checkpoint: epoch {best['epoch']} | widths {WIDTHS} | lr {args['lr']} | "
      f"{args['n']} training puzzles per constellation")

## 1. A puzzle (unseen: val 300+)

In [ ]:
def show_puzzle(images, marks={}, s=104, g=6):
    W, H = 7 * s + 9 * g + 40, 3 * s + 4 * g + 22
    img = Image.new("RGB", (W, H), (236, 238, 242)); d = ImageDraw.Draw(img)
    d.text((g, 4), "context", fill=(60, 64, 72)); d.text((3 * s + 4 * g + 40, 4), "candidates", fill=(60, 64, 72))
    for p in range(9):
        x, y = g + p % 3 * (s + g), 22 + p // 3 * (s + g)
        if p < 8: img.paste(Image.fromarray(images[p]).resize((s, s)).convert("RGB"), (x, y))
        else: d.rectangle([x, y, x + s, y + s], outline=(90, 94, 102), width=2); d.text((x + s // 2 - 3, y + s // 2 - 6), "?", fill=(90, 94, 102))
    for c in range(8):
        x, y = 3 * s + 4 * g + 40 + c % 4 * (s + g), 22 + c // 4 * (s + 18 + g)
        img.paste(Image.fromarray(images[8 + c]).resize((s, s)).convert("RGB"), (x, y))
        rgb, label = marks.get(c, ((60, 64, 72), ""))
        if c in marks: d.rectangle([x - 3, y - 3, x + s + 2, y + s + 2], outline=rgb, width=4)
        d.text((x, y + s + 3), f"{c} {label}", fill=rgb)
    display(img)

CFG, IDX = "distribute_four", 303             # a fixed unseen puzzle, not picked for being solved
stem = f"{DATA}/{CFG}/RAVEN_{IDX}_val"
images, answer = np.load(stem + ".npz")["image"], int(np.load(stem + ".npz")["target"])
panels = B.parse(stem + ".xml", CFG)
print(f"RAVEN_{IDX}_val | {CFG} | answer: candidate {answer}")
show_puzzle(images)

## 2. The graph of this puzzle

**Nodes**
* *Facets*: each object becomes 3 nodes, its type, its size and its colour (thermometer codes: value v = first v of 10 bits on).
* *Panel nodes*: per panel, component and attribute (type, size, colour, number): a summary of that component's objects.
* *Row nodes*: per complete row (1 and 2), component and attribute: "how the attribute changes along the row".
* *Answer nodes*: per component and attribute. Positions have their own chain (panel occupancy → row → answer).

**Edges**
* facet ↔ facet: same attribute, objects at the **same position** in another column of the same row (weight per column pair);
* facet ↔ its panel node (both ways); panel nodes → row node (weight per column); rows 1–2 and panels (3,1), (3,2) → answer.
* Never: between attributes, between rows, or between objects at different positions.

All of this is written once, as rules; grounding instantiates it for the objects present. The structural
atoms (`obj`, `samepos`, `row`, ...) are *hidden*: they decide which neurons exist but add nothing to a sum
(visible, each added 1.0 to every component and hid the positions from the model). Below: the
objects of this puzzle, each drawn as its three facets, and the same-position edges.

In [ ]:
def draw_graph(panels, s=210, g=26):
    img = Image.new("RGB", (3 * s + 4 * g, 3 * s + 4 * g + 24), (255, 255, 255)); d = ImageDraw.Draw(img)
    xy = lambda p, o: (g + (p % 3) * (s + g) + o["cx"] * s, g + (p // 3) * (s + g) + o["cy"] * s)
    for p in range(9):
        x0, y0 = g + (p % 3) * (s + g), g + (p // 3) * (s + g)
        d.rectangle([x0, y0, x0 + s, y0 + s], outline=(180, 178, 169), width=1)
        d.text((x0 + 4, y0 + 3), "?" if p == 8 else f"({p // 3 + 1},{p % 3 + 1})", fill=(136, 135, 128))
    n_edges = 0
    for row in ((0, 1, 2), (3, 4, 5), (6, 7)):
        for a in row:
            for b in row:
                if a < b:
                    for oa in panels[a]:
                        for ob in panels[b]:
                            if oa["pos"] == ob["pos"]:
                                d.line([xy(a, oa), xy(b, ob)], fill=(175, 169, 236), width=2); n_edges += 1
    colours = {"type": (83, 74, 183), "size": (29, 158, 117), "color": (216, 90, 48)}
    for p in range(8):
        for o in panels[p]:
            x, y = xy(p, o)
            for j, a in enumerate(B.FACET_ATTRS):
                cx = x + (j - 1) * 13
                d.ellipse([cx - 6, y - 6, cx + 6, y + 6], fill=colours[a], outline=(255, 255, 255))
    lx = g
    for a, c in colours.items():
        d.ellipse([lx, 3 * s + 3 * g + 10, lx + 10, 3 * s + 3 * g + 20], fill=c); d.text((lx + 14, 3 * s + 3 * g + 9), a, fill=(60, 64, 72)); lx += 70
    d.line([lx, 3 * s + 3 * g + 15, lx + 26, 3 * s + 3 * g + 15], fill=(175, 169, 236), width=2)
    d.text((lx + 32, 3 * s + 3 * g + 9), "same position, another column", fill=(60, 64, 72))
    display(img)
    return n_edges

n_obj = sum(len(panels[p]) for p in range(8))
n_edges = draw_graph(panels)
comps = B.comps(CFG)
print(f"{n_obj} objects -> {3 * n_obj} facet nodes; {n_edges} same-position pairs (an edge each way); "
      f"components {comps}; panel nodes 8 x {len(comps)} x 4; row nodes 2 x {len(comps)} x 4 (+ positions)")
for p in range(8):
    print(f"panel ({p // 3 + 1},{p % 3 + 1}): " + "  ".join(
        f"[pos {o['pos']}: type {o['type']} size {o['size']} colour {o['color']}]" for o in panels[p]))

## 3. The template, widths 32 → 64 → 128

The rules below are the whole model. `f0` projects each facet's code to 32, facet layer 1 is 64 wide, layer 2
and everything above it (panel values `vF`, row relation `rel`, context `ctx`, answer `ans`) 128. Where a
layer is wider than the one below, its residual is a learned projection. Attribute and component are
variables in every rule, so one set of weights serves type, size, colour and number, and both components.

In [ ]:
model = B.build_facet_template(32, 2, shared=not args["per_attr_weights"], rows=not args["no_rows"], widths=WIDTHS,
                               hidden=not args.get("visible_selectors", False))
print(model)

In [ ]:
model.build(Settings(optimizer=Adam(lr=args["lr"]), error_function=MSE(), iso_value_compression=False))
model.load(f"{CKPT}/template_best.bin")
weights = model.state_dict()["weights"] if "weights" in model.state_dict() else model.state_dict()
print("learned weight tensors:", len(weights), "| numbers:", sum(int(np.size(v)) for v in weights.values()))

## 4. Generating the answer

Every puzzle of val 300–399 in all seven constellations is grounded in one go (PyNeuraLogic builds a
model's dataset once); our puzzle is among them. One sample per component of the missing panel.

In [ ]:
cfgs = list(B.POS_MAP)
data = B.load(DATA, cfgs, "val", 300, 100)
units, samples = [], []
for i, (pz, y, cfg) in enumerate(data):
    for k in B.comps(cfg):
        gid = f"v{i}{k}"
        target = B.facet_target(B.comp_objs(pz[8 + y], cfg, k))
        facts = B.puzzle_facts(pz, gid) + B.facet_facts(pz, "therm", cfg) + [R.qcomp(gid, k)]
        units.append((i, k)); samples.append(Sample(R.out(gid)[target.tolist()], facts))
built = list(model.build_dataset(Dataset(samples)))
outs = [dict() for _ in data]
for (i, k), o in zip(units, model.test(built)):
    outs[i][k] = np.asarray(o, dtype=float).flatten()
print(len(data), "puzzles,", len(built), "grounded samples")

def readable(spec):
    vals = lambda a: [v for v, c in enumerate(spec[a]) for _ in range(int(c))]
    return f"positions {list(map(int, spec['pos']))}  type {vals('type')}  size {vals('size')}  colour {vals('color')}"

ex = cfgs.index(CFG) * 100 + (IDX - 300)
for k, o in outs[ex].items():
    print(f"component {k}\n  generated: {readable(B.facet_decode(o))}"
          f"\n  answer:    {readable(B.decode(B.describe(B.comp_objs(panels[8 + answer], CFG, k))))}")

In [ ]:
dist = np.array([sum(np.sum((o - B.facet_target(B.comp_objs(panels[8 + c], CFG, k))) ** 2)
                    for k, o in outs[ex].items()) for c in range(8)])
pick = int(np.argmin(dist))
for c in np.argsort(dist):
    print(f"candidate {c}: distance {dist[c]:7.3f} " + "#" * int(40 * dist[c] / dist.max()) + ("   <- answer" if c == answer else ""))
print(f"picked candidate {pick} | answer {answer} | {'correct' if pick == answer else 'wrong'}")
show_puzzle(images, {answer: ((29, 158, 117), "answer"), **({} if pick == answer else {pick: ((216, 90, 48), "picked")})})

## 5. Accuracy on these 700 unseen puzzles, and the training history

In [ ]:
mse, exact, matched, ties, per = B.evaluate(outs, data, "therm")
print(f"{'constellation':40s} matched   (100 puzzles each)")
for cfg, (h, t) in per.items():
    print(f"{cfg:40s} {100 * h / t:6.1f}%")
print(f"{'all':40s} {100 * matched / len(data):6.1f}%   | generated exactly: {100 * exact['all'] / len(data):.1f}%"
      f" (type {100 * exact['type'] / len(data):.0f}, size {100 * exact['size'] / len(data):.0f},"
      f" colour {100 * exact['color'] / len(data):.0f}, positions {100 * exact['pos'] / len(data):.0f})")

In [ ]:
hist = json.load(open(f"{CKPT}/history.json"))
pct = lambda h, s: 100 * h[s]["matched"] / sum(t for _, t in h[s]["per_constellation"].values())
short = {c: c.replace("_center_single", "").replace("center_single", "center")[:10] for c in cfgs}
print("epoch | train  val0-299  val300+ | " + " ".join(f"{short[c]:>10s}" for c in cfgs) + "   (matched %, val 300-1299)")
for h in hist:
    print(f"{h['epoch']:5d} | {pct(h, 'train'):5.1f}  {pct(h, 'sel'):8.1f}  {pct(h, 'unseen'):7.1f} | " + " ".join(
        f"{100 * h['unseen']['per_constellation'][c][0] / h['unseen']['per_constellation'][c][1]:10.1f}" for c in cfgs))
print(f"best epoch so far (val 0-299): {best['epoch']}")

**Reading the numbers.** Inputs are XML ground-truth attributes (perception is assumed perfect);
accuracies are on validation puzzles, the test split is untouched. *Matched* = the candidate closest to the
generated answer is the correct one (chance 12.5%); *generated exactly* = every component's decoded
positions and value counts equal the answer's. Earlier results (val 300–1299): facet model dim 32 on all
constellations 83.8%; widths 32 → 64 → 128 on distribute_nine alone about 91% (dim 32: 78%).